In [ ]:
!pip install --index-url https://download.pytorch.org/whl/cu121 torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0 peft==0.12.0 evaluate==0.4.2 seqeval==1.2.2 scikit-learn==1.5.2 huggingface_hub==0.24.6 ipywidgets==8.1.3 pandas==2.2.2 numpy==1.26.4 matplotlib==3.9.0 uc-micro-py==1.0.3

In [1]:
import os, random
import torch
import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))
    print("Compute capability (approx via arch):", torch.cuda.get_device_capability(0))
    print("Torch CUDA version:", torch.version.cuda)
print("Torch version:", torch.__version__)


CUDA available: True
CUDA device count: 1
Device name: NVIDIA GeForce RTX 4060 Ti
Compute capability (approx via arch): (8, 9)
Torch CUDA version: 12.1
Torch version: 2.4.1+cu121


In [2]:
import transformers, datasets, accelerate, peft, evaluate, sklearn, huggingface_hub, seqeval
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("evaluate:", evaluate.__version__)
print("scikit-learn:", sklearn.__version__)
import pandas as pd, numpy as np
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

dtype_bf16 = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else None
dtype_fp16 = torch.float16 if torch.cuda.is_available() else None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = torch.nn.Sequential(
    torch.nn.Linear(768, 1024),
    torch.nn.GELU(),
    torch.nn.Linear(1024, 2)
).to(device)

x = torch.randn(16, 768, device=device)

use_bf16 = bool(dtype_bf16)
use_fp16 = (not use_bf16) and bool(dtype_fp16)

if use_bf16:
    autocast_dtype = torch.bfloat16
elif use_fp16:
    autocast_dtype = torch.float16
else:
    autocast_dtype = None

print("AMP dtype selected:", "bf16" if use_bf16 else ("fp16" if use_fp16 else "None"))

if autocast_dtype is not None:
    scaler = torch.cuda.amp.GradScaler(enabled=(autocast_dtype==torch.float16))
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)
    with torch.cuda.amp.autocast(dtype=autocast_dtype):
        y = model(x)
        loss = y.mean()
    if autocast_dtype==torch.float16:
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update()
    else:
        loss.backward(); optim.step()
else:
    y = model(x); loss = y.mean(); loss.backward()

print("Forward/backward with AMP test: OK, loss=", float(loss.detach().cpu()))

transformers: 4.44.2
datasets: 2.21.0
accelerate: 0.33.0
peft: 0.12.0
evaluate: 0.4.2
scikit-learn: 1.5.2
pandas: 2.2.2
numpy: 1.26.4
AMP dtype selected: bf16


C:\Users\GitiAI\AppData\Local\Temp\ipykernel_11040\3557725395.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(autocast_dtype==torch.float16))
C:\Users\GitiAI\AppData\Local\Temp\ipykernel_11040\3557725395.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=autocast_dtype):


Forward/backward with AMP test: OK, loss= -0.034912109375


In [3]:
import os, json, random
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
import numpy as np

DATA_DIR = "data/NER-Models"
SEED = 42
random.seed(SEED)

files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".txt")], key=lambda x: int(x.split(".")[0]))
print(f"Total files: {len(files)}")

def read_file(path):
    tokens, labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                token, label = parts[0], parts[-1]
                label = int(label) if label in ["0","1"] else 0
                tokens.append(token)
                labels.append(label)
    return tokens, labels

all_tokens, all_labels = [], []
for fname in files:
    tks, lbs = read_file(os.path.join(DATA_DIR, fname))
    if len(tks) > 0:
        all_tokens.append(tks)
        all_labels.append(lbs)

print(f"Loaded {len(all_tokens)} samples.")

train_toks, temp_toks, train_lbls, temp_lbls = train_test_split(all_tokens, all_labels, test_size=0.2, random_state=SEED)
val_toks, test_toks, val_lbls, test_lbls = train_test_split(temp_toks, temp_lbls, test_size=0.5, random_state=SEED)

train_ds = Dataset.from_dict({"tokens": train_toks, "labels": train_lbls})
val_ds = Dataset.from_dict({"tokens": val_toks, "labels": val_lbls})
test_ds = Dataset.from_dict({"tokens": test_toks, "labels": test_lbls})

dataset = DatasetDict({"train": train_ds, "validation": val_ds, "test": test_ds})
print(dataset)
print("Sample example:\n", dataset["train"][0])

def label_distribution(labels):
    flat = np.concatenate(labels)
    zeros = int(np.sum(flat == 0))
    ones = int(np.sum(flat == 1))
    total = int(len(flat))
    return {
        "0 (O)": zeros,
        "1 (ENT)": ones,
        "ratio (ENT%)": round((ones / total) * 100, 2)
    }

dist_train = label_distribution([np.array(x) for x in dataset["train"]["labels"]])
dist_val = label_distribution([np.array(x) for x in dataset["validation"]["labels"]])
dist_test = label_distribution([np.array(x) for x in dataset["test"]["labels"]])

print("Train distribution:", dist_train)
print("Validation distribution:", dist_val)
print("Test distribution:", dist_test)

os.makedirs("reports", exist_ok=True)

def ensure_serializable(d):
    return {k: int(v) if isinstance(v, (np.integer,)) else v for k, v in d.items()}

with open("reports/dataset_stats.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "train": ensure_serializable(dist_train),
            "val": ensure_serializable(dist_val),
            "test": ensure_serializable(dist_test)
        },
        f,
        ensure_ascii=False,
        indent=2
    )

Total files: 18269
Loaded 18269 samples.
DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 14615
    })
    validation: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1827
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1827
    })
})
Sample example:
 {'tokens': ['حالا', 'دراین', 'اگر', 'این', 'زمین', 'به', 'فرد', 'دیگری', 'هم', 'واگذار', 'شده', 'باشند', '،', 'برای', 'ما', 'فرقی', 'نمی', 'که', 'زمین', 'متعلق', 'چه', 'کسی', 'است', 'زیرا', 'ما', 'قرارداد', 'را', 'فسخ', 'کرده', '.'], 'labels': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
Train distribution: {'0 (O)': 451881, '1 (ENT)': 10835, 'ratio (ENT%)': 2.34}
Validation distribution: {'0 (O)': 55954, '1 (ENT)': 1157, 'ratio (ENT%)': 2.03}
Test distribution: {'0 (O)': 56840, '1 (ENT)': 1371, 'ratio (ENT%)': 2.36}


In [7]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification

MODEL_NAME = "HooshvareLab/bert-fa-base-uncased"
MAX_LEN = 256 
SEED = 42

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded:", MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LEN,
        padding=False
    )

    labels = []
    for i, label_seq in enumerate(examples["labels"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_seq[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

Tokenizer loaded: HooshvareLab/bert-fa-base-uncased


In [9]:
from datasets import DatasetDict
import numpy as np


tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    num_proc=1,
    remove_columns=["tokens", "labels"],
    desc="Tokenizing and aligning labels"
)

data_collator = DataCollatorForTokenClassification(
    tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt"
)


sample = tokenized_datasets["train"][0]
print("Keys:", sample.keys())
print("Input IDs length:", len(sample["input_ids"]))
print("Labels length:", len(sample["labels"]))


num_ignore = np.sum(np.array(sample["labels"]) == -100)
print(f"Ignored tokens (mask -100): {num_ignore}/{len(sample['labels'])} = {round(num_ignore/len(sample['labels'])*100,2)}%")


Tokenizing and aligning labels:   0%|          | 0/14615 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/1827 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/1827 [00:00<?, ? examples/s]

Keys: dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])
Input IDs length: 32
Labels length: 32
Ignored tokens (mask -100): 2/32 = 6.25%


In [10]:
from transformers import AutoModelForTokenClassification
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

id2label = {0: "NON-ENT", 1: "ENT"}
label2id = {"NON-ENT": 0, "ENT": 1}

base_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)
print("Base model loaded.")


pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

C:\Users\GitiAI\anaconda3\envs\peft-ner\lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GitiAI\.cache\huggingface\hub\models--HooshvareLab--bert-fa-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some weights of BertForTokenClassification were not initialized from the model checkpoint 

Base model loaded.


In [12]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    task_type="TOKEN_CLS",
    target_modules=["query", "key", "value", "dense"]
)


model = get_peft_model(base_model, lora_config)
print("LoRA adapters applied.")


LoRA adapters applied.


In [13]:
def print_trainable_parameters(model):
    trainable, total = 0, 0
    for _, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Trainable params: {trainable:,} / {total:,} "
          f"({trainable/total*100:.2f}% trainable)")

print_trainable_parameters(model)


Trainable params: 2,655,746 / 164,908,036 (1.61% trainable)


In [15]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

sample = tokenized_datasets["train"][0]
input_ids = torch.tensor([sample["input_ids"]], device="cuda")
attention_mask = torch.tensor([sample["attention_mask"]], device="cuda")

with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
print("Forward test OK. Output logits shape:", outputs.logits.shape)



Forward test OK. Output logits shape: torch.Size([1, 32, 2])


C:\Users\GitiAI\anaconda3\envs\peft-ner\lib\site-packages\transformers\models\bert\modeling_bert.py:439: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [22]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

 
    true_labels, true_preds = [], []
    for pred, label in zip(preds, labels):
        mask = label != -100
        true_labels.extend(label[mask])
        true_preds.extend(pred[mask])

    precision = precision_score(true_labels, true_preds, average="binary")
    recall = recall_score(true_labels, true_preds, average="binary")
    f1 = f1_score(true_labels, true_preds, average="binary")

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }

training_args = TrainingArguments(
    output_dir="outputs/checkpoints",
    evaluation_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=200,
    save_steps=200,
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    gradient_accumulation_steps=1,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [23]:
train_result = trainer.train()
trainer.save_model("outputs/adapters/") 
trainer.save_state()

metrics = train_result.metrics
metrics["train_samples"] = len(tokenized_datasets["train"])
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
print("Training completed.")


Step,Training Loss,Validation Loss,Precision,Recall,F1
200,No log,0.010376,0.938300,0.920500,0.929300
400,No log,0.008116,0.939900,0.945500,0.942700
600,0.009900,0.008507,0.917400,0.941200,0.929200
800,0.009900,0.007416,0.946600,0.934300,0.940400


***** train metrics *****
  epoch                    =     0.8753
  total_flos               =   512062GF
  train_loss               =     0.0094
  train_runtime            = 0:02:11.14
  train_samples            =      14615
  train_samples_per_second =     557.21
  train_steps_per_second   =     34.847
Training completed.


In [24]:
metrics_val = trainer.evaluate(tokenized_datasets["validation"])
trainer.log_metrics("validation", metrics_val)
trainer.save_metrics("validation", metrics_val)

print("Validation metrics:", metrics_val)


***** validation metrics *****
  epoch                   =     0.8753
  eval_f1                 =     0.9427
  eval_loss               =     0.0081
  eval_precision          =     0.9399
  eval_recall             =     0.9455
  eval_runtime            = 0:00:03.72
  eval_samples_per_second =    490.544
  eval_steps_per_second   =     15.573
Validation metrics: {'eval_loss': 0.008116108365356922, 'eval_precision': 0.9399, 'eval_recall': 0.9455, 'eval_f1': 0.9427, 'eval_runtime': 3.7244, 'eval_samples_per_second': 490.544, 'eval_steps_per_second': 15.573, 'epoch': 0.87527352297593}


In [25]:
import json
from sklearn.metrics import confusion_matrix

metrics_test = trainer.evaluate(tokenized_datasets["test"])
print("Test metrics:", metrics_test)


os.makedirs("reports", exist_ok=True)
with open("reports/metrics_test.json", "w", encoding="utf-8") as f:
    json.dump(metrics_test, f, ensure_ascii=False, indent=2)


Test metrics: {'eval_loss': 0.007628475781530142, 'eval_precision': 0.9503, 'eval_recall': 0.9475, 'eval_f1': 0.9489, 'eval_runtime': 4.121, 'eval_samples_per_second': 443.335, 'eval_steps_per_second': 14.074, 'epoch': 0.87527352297593}


In [26]:
import numpy as np

predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions, axis=-1)

y_true, y_pred = [], []
for p, l in zip(preds, labels):
    mask = l != -100
    y_true.extend(l[mask])
    y_pred.extend(p[mask])

cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix (rows=true, cols=pred):\n", cm)

for i in range(3):
    tokens = tokenized_datasets["test"][i]["input_ids"]
    words = tokenizer.convert_ids_to_tokens(tokens)
    preds_i = preds[i][:len(words)]
    print("\nSample", i + 1)
    print([(w, int(p)) for w, p in zip(words[:20], preds_i[:20])])


Confusion Matrix (rows=true, cols=pred):
 [[56772    68]
 [   72  1299]]

Sample 1
[('[CLS]', 0), ('البته', 0), ('تاکنون', 0), ('این', 0), ('براندازی', 0), ('مدرن', 0), ('با', 0), ('ابزار', 0), ('اطلاعرسانی', 0), ('و', 0), ('در', 0), ('ظاهری', 0), ('مسالمتامیز', 0), ('و', 0), ('با', 0), ('تبدیل', 0), ('قلمها', 0), ('به', 0), ('مسلسل', 0), ('!', 0)]

Sample 2
[('[CLS]', 0), ('درست', 0), ('است', 0), ('که', 0), ('منافع', 0), ('ملی', 0), ('اولویت', 0), ('دارد', 0), ('اما', 0), ('واقعا', 0), ('در', 0), ('این', 0), ('شرایط', 0), ('کار', 0), ('کردن', 0), ('سخت', 0), ('است', 0), ('.', 0), ('[SEP]', 0)]

Sample 3
[('[CLS]', 0), ('این', 0), ('فیلم', 0), ('جهت', 0), ('برنامههای', 0), ('مبارزه', 0), ('با', 0), ('دوپینگ', 0), ('کمیته', 0), ('بینالمللی', 0), ('المپیک', 0), ('ساخته', 0), ('میشود', 0), ('و', 0), ('قرار', 0), ('است', 0), ('در', 0), ('طول', 0), ('برگزاری', 0), ('بازیهای', 0)]


In [27]:
summary = {
    "validation": {
        "precision": 0.9399,
        "recall": 0.9455,
        "f1": 0.9427,
        "loss": 0.0081,
    },
    "test": {
        "precision": round(metrics_test["eval_precision"], 4),
        "recall": round(metrics_test["eval_recall"], 4),
        "f1": round(metrics_test["eval_f1"], 4),
        "loss": round(metrics_test["eval_loss"], 4),
    }
}

with open("reports/summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Summary report saved to reports/summary.json")


Summary report saved to reports/summary.json


In [28]:
adapter_dir = "outputs/adapters/final_lora_model"
model.save_pretrained(adapter_dir)

tokenizer.save_pretrained(adapter_dir)

print(f"LoRA adapter saved at: {adapter_dir}")


LoRA adapter saved at: outputs/adapters/final_lora_model


In [30]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForTokenClassification, AutoTokenizer

adapter_dir = "outputs/adapters/final_lora_model"
peft_config = PeftConfig.from_pretrained(adapter_dir)
label_list = ["NON-ENT", "ENT"]
base_model = AutoModelForTokenClassification.from_pretrained(peft_config.base_model_name_or_path, num_labels=len(label_list))
model = PeftModel.from_pretrained(base_model, adapter_dir)
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

print("LoRA adapter loaded and model reconstructed successfully.")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at HooshvareLab/bert-fa-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LoRA adapter loaded and model reconstructed successfully.


In [31]:
import torch

text = "ریما کوسا تصویرگری کرده است."
tokens = tokenizer(text.split(), is_split_into_words=True, return_tensors="pt", padding=True, truncation=True)
tokens = {k: v.to(model.device) for k, v in tokens.items()}

with torch.no_grad():
    outputs = model(**tokens)
preds = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

decoded_labels = [label_list[p] for p in preds]
print(list(zip(text.split(), decoded_labels)))


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[('ریما', 'NON-ENT'), ('کوسا', 'ENT'), ('تصویرگری', 'ENT'), ('کرده', 'ENT'), ('است.', 'NON-ENT')]
